# 03. Adam Optimizer Demystified: From SGD to Adaptive Moments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/The-Pocket/PocketFlow-Tutorial-Video-Generator/blob/main/docs/llm/03_Adam_Optimizer_Demystified.ipynb)

> **Tutorial Overview**: Understand why standard SGD struggles in ravines and saddle points, how Momentum solves oscillations, how RMSprop scales learning rates, and how Adam combines both with Bias Correction.
> **Original Source**: `adam.md`

---


In [ ]:
# Setup & Imports
!pip install -q torch matplotlib numpy

import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)


# **Title: Give me 30 min, I will make the Adam Optimizer Click. Forever.**

## Intro

You know Gradient Descent. You know the formula: `new_weight = old_weight - learning_rate * gradient`. It's simple. It works.

But it's not what powers modern AI.

In every state-of-the-art model, in every high-performance training script, you see the same name: **Adam**. You're told to just use it. It's the default. It's "better."

But why?

You look up the algorithm and are hit with a wall of math. A black box of Greek letters and strange terms that feel impossibly complex.

*   Exponentially Weighted Moving Average (EWMA)
*   First and Second Moments
*   Bias Correction

It seems like a magic spell you're supposed to cast without understanding.

Here's the secret: **Adam isn't one complex idea. It's three simple ideas, bolted together to solve three specific problems.**

Give me 30 minutes. We will tear Adam down to its fundamental parts and rebuild it from the ground up. No skipped steps. No magic. By the end, you will have a deep, intuitive, and permanent understanding of every single component.

This is the algorithm you are about to master:

1.  **First Moment (Momentum):** $m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$
2.  **Second Moment (Adaptive LR):** $v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$
3.  **Bias Correction:** $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$ and $\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$
4.  **The Update:** $\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

That wall of math will look simple. You will see *why* each piece exists and the exact problem it solves.

Ready to make it click? Let's begin.

## **Part 1: The First Problem - Inefficient Progress**

**The Core Problem:** Gradient Descent is memoryless. As it gets closer to the minimum and the gradient shrinks, its steps become smaller and smaller, causing it to slow down dramatically.

Let's use the simple function `f(p) = p**2`. The minimum is at `p=0`, and the gradient is `f'(p) = 2p`. We start at `p=10` and use a learning rate of `η = 0.1`.

#### **Algorithm 1: Standard Gradient Descent**

The update rule is simple and direct.


```python
FOR each iteration:
  gradient = 2 * params
  params = params - 0.1 * gradient
```


This is our baseline—the slow, steady crawl.

| Iteration | Current `p` | Gradient `g = 2p` | Update `0.1 * g` | New `p` |
| :-------- | :---------- | :---------------- | :--------------- | :------ |
| 0         | 10.000      | 20.000            | 2.000            | 8.000   |
| 1         | 8.000       | 16.000            | 1.600            | 6.400   |
| 2         | 6.400       | 12.800            | 1.280            | 5.120   |
| 3         | 5.120       | 10.240            | 1.024            | 4.096   |
| 4         | 4.096       | 8.192             | 0.819            | 3.277   |

**Analysis of the Slowness:** The "Update" size is constantly shrinking: `2.0` → `1.6` → `1.28`... This is **deceleration**. The algorithm becomes less effective with every step.

#### **Algorithm 2: Gradient Descent with Momentum**

Now, let's add a `velocity` term with a more moderate `beta` of `0.5`. This will allow inertia to build up without running out of control.


```python
velocity = 0
FOR each iteration:
  gradient = 2 * params
  velocity = 0.5 * velocity + gradient
  params = params - 0.1 * velocity
```


Watch the difference in convergence.

| Iteration | Current `p` | Gradient `g` | Velocity `v = 0.5*v + g` | New `p` |
| :-------- | :---------- | :----------- | :--------------------------- | :------ |
| 0         | 10.000      | 20.000       | `0.5*0 + 20.0 = 20.000`      | 8.000   |
| 1         | 8.000       | 16.000       | `0.5*20.0 + 16.0 = 26.000`   | 5.400   |
| 2         | 5.400       | 10.800       | `0.5*26.0 + 10.8 = 23.800`   | 3.020   |
| 3         | 3.020       | 6.040        | `0.5*23.8 + 6.04 = 17.940`   | 1.226   |
| 4         | 1.226       | 2.452        | `0.5*17.94 + 2.45 = 11.422`  | 0.084   |

**Analysis of the Success:**
*   **Compare `p` at Iteration 4:** Standard Gradient Descent is still far away at `3.277`. Momentum is already at `0.084`, practically at the minimum. This is a clear, unambiguous win.
*   **Look at the `velocity`:** In step 1, the gradient was `16`, but the velocity was `26`. In step 2, the gradient was `10.8`, but the velocity was `23.8`. Because the gradients were all in the same direction, they accumulated, creating a much larger and more effective update step. This is **controlled acceleration**.
*   **No Instability:** Unlike the previous bad example, this version converges beautifully without any wild overshooting.

---
#### **Revisiting the Ravine: The Two Jobs of Momentum**

Now we can confidently state that Momentum is a superior algorithm. In a complex landscape like our 2D ravine (`f(p) = p[0]**2 + 50 * p[1]**2`), it performs two critical jobs simultaneously:

1.  **Accelerates:** In the shallow `p[0]` direction, the gradients are small but consistent. Momentum builds up velocity here—just like in our successful 1D example—speeding up progress along the valley floor.
2.  **Damps:** In the steep `p[1]` direction, the gradients are huge but constantly flip signs (`+150`, `-120`, etc.). When Momentum averages these opposing forces, they cancel each other out, which powerfully suppresses the wasteful zig-zagging.

Momentum intelligently uses its memory of past gradients to navigate more efficiently.

**Problem Solved:** We have a mechanism to fix Gradient Descent's inefficient, memoryless updates.

**But a new problem emerges:** While smarter, this approach still applies the same learning rate to every parameter. Isn't there a way to give each parameter its *own* adaptive learning rate from the start?

## **Part 2: The Second Problem - Inflexible Learning Rates**

**The Core Problem:** Momentum helps find a better direction, but it's still handicapped by a single, global learning rate. This fails when parameters have vastly different sensitivities.

Let's design a function where this failure is guaranteed:
`f(p) = 50 * p[0]**2 + p[1]**2`

The minimum is at `(0, 0)`. The gradient vector is:
*   `∂f/∂p[0] = 100 * p[0]`
*   `∂f/∂p[1] = 2 * p[1]`

The gradient for `p[0]` is **50 times stronger** than for `p[1]`. This means `p[0]` is an extremely "sensitive" parameter, while `p[1]` is "stubborn."

#### **Algorithm 1: Naive Gradient Descent**

To prevent the update for the sensitive `p[0]` from exploding, we are forced to choose a tiny learning rate. Let's use `η = 0.01`. We will start at `p = (1.5, 10.0)`.


```python
FOR each iteration:
  gradient = [100*p[0], 2*p[1]]
  params = params - 0.01 * gradient
```


Watch how slowly `p[1]` converges.

| Iteration | Current `p` | Gradient `g` | Update `0.01 * g` | New `p` |
| :-------- | :---------- | :------------- | :------------------ | :---------- |
| 0         | `(1.500, 10.000)` | `[150.0, 20.0]`| `[1.500, 0.200]`    | `(0.000, 9.800)` |
| 1         | `(0.000, 9.800)`  | `[0.0, 19.6]`  | `[0.000, 0.196]`    | `(0.000, 9.604)` |
| 2         | `(0.000, 9.604)`  | `[0.0, 19.208]`| `[0.000, 0.192]`    | `(0.000, 9.412)` |
| 3         | `(0.000, 9.412)`  | `[0.0, 18.824]`| `[0.000, 0.188]`    | `(0.000, 9.224)` |

**Analysis of the Failure:** The learning rate `η=0.01` was just right for `p[0]`, which converged in one step. But this same learning rate is cripplingly small for `p[1]`. Its progress is a slow crawl: `10.0` → `9.8` → `9.6` → `9.4`. It will take hundreds of steps to reach zero. This is the definition of inefficiency.

#### **Algorithm 2: AdaGrad (Adaptive Gradient)**

AdaGrad gives each parameter its own learning rate that adapts over time. It does this by dividing the base learning rate by the square root of the sum of all past squared gradients for that parameter.

**THE ALGORITHM: AdaGrad**


```python
g_squared = [0, 0]
FOR each iteration:
  gradient = calculate_gradient(params)
  g_squared += gradient**2
  adapted_lr = learning_rate / (sqrt(g_squared) + epsilon)
  params = params - adapted_lr * gradient
```


Because it's adaptive, we can use a much more aggressive base learning rate. Let's use `η = 1.5`.

| Iteration | Current `p` | Gradient `g` | `g_squared` (Accumulator) | Effective LR `η/sqrt(g_sq)` | Update | New `p` |
| :-------- | :---------- | :------------- | :------------------------ | :------------------------- | :------- | :---------- |
| 0         | `(1.5, 10.0)` | `[150, 20]`    | `[22500, 400]`            | `[0.01, 0.075]`            | `[1.5, 1.5]` | `(0.0, 8.5)` |
| 1         | `(0.0, 8.5)`  | `[0, 17]`      | `[22500, 689]`            | `[0.01, 0.057]`            | `[0, 0.97]`| `(0.0, 7.53)` |
| 2         | `(0.0, 7.53)` | `[0, 15.06]`   | `[22500, 916]`            | `[0.01, 0.050]`            | `[0, 0.75]`| `(0.0, 6.78)` |
| 3         | `(0.0, 6.78)` | `[0, 13.56]`   | `[22500, 1100]`           | `[0.01, 0.045]`            | `[0, 0.61]`| `(0.0, 6.17)` |

**Analysis of the Success:**
*   **Look at `p[0]`:** In step 0, the accumulated `g_squared[0]` was `22500`. Its square root is `150`. The effective learning rate for `p[0]` became `1.5 / 150 = 0.01`. AdaGrad *automatically* discovered the perfect small learning rate for the sensitive parameter.
*   **Look at `p[1]`:** In step 0, `g_squared[1]` was only `400`. Its square root is `20`. The effective learning rate for `p[1]` was `1.5 / 20 = 0.075`. This is much larger than the `0.01` used by naive GD.
*   **The Final Comparison:** After 4 steps, naive Gradient Descent got `p[1]` to `9.224`. AdaGrad got it to `6.17`. AdaGrad is converging dramatically faster because it assigned a more appropriate learning rate to the stubborn parameter.

**Problem Solved:** We have introduced adaptive, per-parameter learning rates, making optimization robust to wildly different gradient scales.

**But a new problem emerges:** Look at AdaGrad's `g_squared` accumulator. The values `[22500, 400]` grew to `[22500, 1100]`. This sum *only ever increases*. Over a long training run, this denominator will grow so large that the effective learning rate for all parameters will shrink to effectively zero, stopping learning prematurely. This is known as a "decaying learning rate" problem, and it's what Adam must fix next.

## **Part 3: Deconstructing Adam - The Theory**

Our goal is to create an optimizer that combines the directional intelligence of Momentum with the adaptive learning rates of AdaGrad, while fixing AdaGrad's "dying learning rate" problem. Adam achieves this by using a more flexible memory system for both direction and magnitude.

#### **The Complete Adam Algorithm**

Here is the full blueprint of the algorithm we are about to deconstruct.

1.  **Initialize:**
    *   `m = 0` (First moment vector)
    *   `v = 0` (Second moment vector)
    *   `t = 0` (Timestep)

2.  **Loop for each training iteration:**
    *   `t = t + 1`
    *   `g_t =` Calculate gradient at current step
    *   **Update Biased Moment Estimates:**
        *   $m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$
        *   $v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$
    *   **Compute Bias-Corrected Estimates:**
        *   $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$
        *   $\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$
    *   **Update Parameters:**
        *   $\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

Now, let's break down what each part of this machine does.

#### **The Core Component: Exponentially Weighted Moving Average (EWMA)**

Adam is built entirely on the concept of the EWMA, which is a "forgetful" average. Its formula is:

`average_t = β * average_{t-1} + (1 - β) * new_value_t`

*   `β` (beta) is the "decay rate" or memory factor, a number between 0 and 1. It controls how much of the old average to keep.
*   A high `β` (like 0.99) means the average has a long memory and changes slowly.
*   A low `β` (like 0.1) means the average has a short memory and reacts quickly to new values.

This "forgetful" property is what fixes AdaGrad's problem of its learning rate only ever shrinking.

#### **Line-by-Line Breakdown of the Algorithm**

**Line 1: `m_t = β₁ * m_{t-1} + (1 - β₁) * g_t`**
*   **What it is:** The **First Moment Estimate**.
*   **Purpose:** This is the **Direction Engine**. It calculates the EWMA of the gradients (`g_t`). It acts like a more robust version of Momentum's velocity, tracking the average direction of descent.
*   **`β₁` (beta1):** The memory factor for the direction. It is typically set to `0.9`.

**Line 2: `v_t = β₂ * v_{t-1} + (1 - β₂) * g_t²`**
*   **What it is:** The **Second Moment Estimate**.
*   **Purpose:** This is the **Adaptive Learning Rate Engine**. It calculates the EWMA of the *squared* gradients (`g_t²`). This tracks the average magnitude of the gradients, replacing AdaGrad's ever-growing sum with a "forgetful" average.
*   **`β₂` (beta2):** The memory factor for the magnitude. It is typically set to `0.999`, giving it a much longer memory than `m` to ensure the learning rate stays stable.

**Line 3 & 4: The Bias Correction (`m_hat`, `v_hat`)**
*   **The Problem:** `m` and `v` are initialized to zero. At the beginning of training, their values are artificially small because they are biased toward this zero starting point. This would cause the optimizer to take tiny, inefficient steps initially.
*   **The Solution:** These lines correct for that initial bias.
    *   `β₁^t` means the constant `β₁` raised to the power of the current timestep `t`.
    *   At `t=1`, this correction is large, boosting the estimates to be more accurate.
    *   As `t` increases, the correction term `(1 - β^t)` approaches 1, and the correction fades away, which is exactly what we need.

**Line 5: The Final Update (`θ_t = ...`)**
*   **What it is:** The actual parameter update step.
*   **Purpose:** This line combines all the components.
    *   It determines the step **direction** using `m_hat`.
    *   It scales the step size for each parameter using `sqrt(v_hat)`. This division is what gives Adam its adaptive, per-parameter learning rate.
    *   `η` is the master learning rate you provide, and `ε` is a tiny value to prevent division by zero.

In essence, Adam runs two intelligent, "forgetful" averages—one for direction and one for magnitude—corrects their initial bias, and then uses them to perform a robust and adaptive update step.

## **Part 4: Adam in Action - A Definitive Example**

**The Scenario**
We will create a situation designed to make naive Gradient Descent fail catastrophically, so we can see how Adam's internal mechanisms save it.

*   **Function:** `f(p) = p**2` (Minimum at `p=0`, Gradient `g = 2p`)
*   **Starting Point:** `p = 10`
*   **Learning Rate (`η`):** We will use an **explosive** learning rate of `η = 1.05`. For this function, any learning rate greater than `1.0` will cause naive Gradient Descent to diverge.

#### **Baseline: Naive Gradient Descent (Complete Failure)**

The update rule is simple: `p_new = p_old - η * g`.

| Iteration | Current `p` | Gradient `g = 2p` | Update `1.05 * g` | New `p` |
| :-------- | :---------- | :---------------- | :---------------- | :------- |
| 0         | 10.00       | 20.00             | 21.00             | -11.00   |
| 1         | -11.00      | -22.00            | -23.10            | 12.10    |
| 2         | 12.10       | 24.20             | 25.41             | -13.31   |
| 3         | -13.31      | -26.62            | -27.95            | 14.64    |
| 4         | 14.64       | 29.28             | 30.74             | -16.10   |

**Analysis of the Failure:** Look at the absolute value of `p`. It is growing at every step: `10` → `11` → `12.1` → `13.31`. This is **divergence**. The algorithm is not just inefficient; it is fundamentally broken and exploding towards infinity. It is unusable with this learning rate.

---

#### **Adam: Taming the Explosive Learning Rate**

Now, we give Adam the **exact same unusable learning rate** (`η = 1.05`) and watch how its machinery handles the situation. We will use standard `β₁=0.9` and `β₂=0.999`. Let's trace the first 10 iterations to see the full story.

| t | `p` | `g` | `m` | `v` | `m_hat` | `v_hat` | Update | New `p` |
|:-:|:----|:----|:----|:----|:----|:----|:--- |:--- |
| 1 | 10.00 | 20.00 | 2.00 | 0.40 | 20.00 | 400.0 | 1.05 | 8.95 |
| 2 | 8.95 | 17.90 | 3.59 | 0.72 | 18.89 | 360.4 | 1.04 | 7.91 |
| 3 | 7.91 | 15.82 | 4.81 | 0.97 | 17.58 | 354.7 | 0.98 | 6.93 |
| 4 | 6.93 | 13.86 | 5.72 | 1.16 | 16.58 | 341.7 | 0.94 | 5.99 |
| 5 | 5.99 | 11.98 | 6.35 | 1.31 | 15.51 | 319.8 | 0.91 | 5.08 |
| 6 | 5.08 | 10.16 | 6.73 | 1.41 | 14.41 | 294.1 | 0.88 | 4.20 |
| 7 | 4.20 | 8.40 | 6.90 | 1.48 | 13.35 | 266.3 | 0.85 | 3.35 |
| 8 | 3.35 | 6.70 | 6.88 | 1.51 | 12.35 | 237.9 | 0.82 | 2.53 |
| 9 | 2.53 | 5.06 | 6.70 | 1.51 | 11.41 | 209.9 | 0.80 | 1.73 |
| 10| 1.73 | 3.46 | 6.38 | 1.48 | 10.53 | 182.9 | 0.78 | 0.95 |

**Analysis of the Definitive Success:**

1.  **Adam is Stable and Converging:** The primary result is undeniable. Where naive GD exploded, Adam's `p` value steadily and rapidly decreases: `10 → 8.95 → ... → 0.95`. It successfully tamed an otherwise unusable learning rate and is converging beautifully.

2.  **The Adaptive Rate is the Hero:** How did it survive? Look at the `v_hat` column. It starts high (`400`) because the initial gradients are large, and then it slowly decays as `p` gets smaller. The crucial term is the denominator of the final update, `sqrt(v_hat)`. At step 1, this was `sqrt(400) = 20`. This means Adam calculated an **"effective learning rate"** of `η / 20 = 1.05 / 20 ≈ 0.053`. It automatically throttled the explosive `1.05` down to a safe and effective `0.053`. This dynamic self-correction is Adam's superpower.

3.  **Momentum Provides the Smoothness:** Look at the `m_hat` column. It provides a smooth, consistent estimate of the direction, preventing the wild oscillations we saw in the failed GD example.

This walkthrough proves Adam's value. It is not just another optimizer; it is a robust, self-correcting system. It takes a potentially dangerous hyperparameter (the learning rate) and adapts it on the fly, protecting the training process from instability and divergence. This robustness is precisely why Adam is the default, go-to optimizer for nearly all modern deep learning applications.

## **Conclusion: From Simple Steps to Intelligent Adaptation**

You have just mastered the core logic behind modern optimization. We began with the simple idea of Gradient Descent and systematically solved its flaws, piece by piece, culminating in Adam.

The journey was a logical progression:

| Problem                                       | Solution                | Key Idea                                                               |
| :-------------------------------------------- | :---------------------- | :--------------------------------------------------------------------- |
| 1. **Inefficient Direction & Oscillation**    | **Momentum**            | Average past gradients to find a better, smoother direction.           |
| 2. **Inflexible, One-Size-Fits-All LR**       | **AdaGrad**             | Give each parameter its own learning rate based on its gradient history. |
| 3. **AdaGrad's LR Dies Prematurely**          | **"Forgetful" Averages (EWMA)** | Replace the permanent sum with a moving average that forgets the past.   |
| 4. **Initial Steps are Too Small**            | **Bias Correction**     | Correct the initial bias of the moving averages for a faster start.    |

**Adam is not a single complex idea; it is the synthesis of these four solutions.** It uses an EWMA of gradients for **direction** (Momentum) and an EWMA of squared gradients for a per-parameter **adaptive rate** (AdaGrad), fixing both with **bias correction**.

The result is a robust, high-performance algorithm that automatically adapts to the unique challenges of a complex loss landscape. You now understand not just *what* Adam does, but *why* every single component of its machinery exists. It's not magic—it's brilliant engineering.


## **Interactive Playground: SGD vs Momentum vs RMSprop vs Adam on a Ravine Surface**
Let's build custom optimizers from scratch and compare their trajectories navigating an elongated ravine (an ill-conditioned quadratic surface $f(x, y) = 0.1 x^2 + 2.0 y^2$).


In [ ]:
# Custom implementation of Adam from scratch
class CustomAdam:
    def __init__(self, params, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                g = p.grad
                # 1. First Moment (Momentum)
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
                # 2. Second Moment (RMSprop)
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (g ** 2)
                # 3. Bias Correction
                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)
                # 4. Update
                p -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

# Define an elongated ravine function
def loss_fn(x, y):
    return 0.1 * (x ** 2) + 2.0 * (y ** 2)

def run_optimizer(opt_name, steps=50):
    point = torch.tensor([-4.0, 3.0], requires_grad=True)
    history = [point.detach().numpy().copy()]
    
    if opt_name == "SGD":
        opt = torch.optim.SGD([point], lr=0.15)
    elif opt_name == "SGD+Momentum":
        opt = torch.optim.SGD([point], lr=0.08, momentum=0.9)
    elif opt_name == "RMSprop":
        opt = torch.optim.RMSprop([point], lr=0.1)
    elif opt_name == "Custom Adam":
        opt = CustomAdam([point], lr=0.2)
        
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(point[0], point[1])
        loss.backward()
        opt.step()
        history.append(point.detach().numpy().copy())
    return np.array(history)

# Run simulations
optimizers = ["SGD", "SGD+Momentum", "RMSprop", "Custom Adam"]
colors = ['red', 'purple', 'green', 'blue']

# Plot contours
X_grid, Y_grid = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
Z_grid = loss_fn(X_grid, Y_grid)

plt.figure(figsize=(10, 7))
plt.contour(X_grid, Y_grid, Z_grid, levels=30, cmap='plasma', alpha=0.6)

for name, col in zip(optimizers, colors):
    traj = run_optimizer(name, steps=60)
    plt.plot(traj[:, 0], traj[:, 1], marker='o', markersize=3, label=name, color=col, lw=2)

plt.plot(0, 0, 'r*', markersize=15, label='Global Minimum (0,0)')
plt.title("Optimization Trajectories in an Elongated Ravine")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
